# 06 — Export formats

**Audience:** Users choosing how to deliver a transformed Premise database to Brightway, Activity Browser, SimaPro, OpenLCA, or another Premise user.

**Prerequisites:** A configured Brightway source project, at least one built scenario, and two scenarios for superstructure export.

**Learning goals:** select one export method, configure its destination, and understand which outputs are portable.


## Outline

1. Build two lightweight scenarios.
2. Choose one export target.
3. Interpret portability and lifecycle constraints.

Matrix exports and Brightway scenario arrays have dedicated tutorials: notebooks 08 and 07 respectively.


In [ ]:
import os
from pathlib import Path

import bw2data as bd

from premise import NewDatabase

PROJECT = "ecoinvent-3.12-cutoff"
SOURCE_DATABASE = "ecoinvent-3.12-cutoff"
BIOSPHERE_DATABASE = "ecoinvent-3.12-biosphere"
PREMISE_KEY = os.environ.get("PREMISE_KEY")
EXPORT_ROOT = Path("export/tutorial-exports")

if not PREMISE_KEY:
    raise RuntimeError("Set PREMISE_KEY before running this tutorial.")
bd.projects.set_current(PROJECT)


In [ ]:
SCENARIOS = [
    {"model": "remind", "pathway": "SSP2-NDC", "year": 2030},
    {"model": "remind", "pathway": "SSP2-NDC", "year": 2050},
]

ndb = NewDatabase(
    scenarios=[scenario.copy() for scenario in SCENARIOS],
    source_db=SOURCE_DATABASE,
    source_version="3.12",
    biosphere_name=BIOSPHERE_DATABASE,
    key=PREMISE_KEY,
)
ndb.update(["electricity"])


## Choose one export

Export methods finalize or serialize substantial state. Set `EXPORT_KIND` once and run only the matching branch for a given `ndb` instance.


In [ ]:
EXPORT_KIND = "brightway"
# Options: brightway, superstructure, simapro, olca, datapackage

if EXPORT_KIND == "brightway":
    names = ["premise-remind-2030", "premise-remind-2050"]
    ndb.write_db_to_brightway(name=names)
elif EXPORT_KIND == "superstructure":
    ndb.write_superstructure_db_to_brightway(
        name="premise-remind-superstructure",
        file_format="csv",
    )
elif EXPORT_KIND == "simapro":
    ndb.write_db_to_simapro(filepath=str(EXPORT_ROOT / "simapro"))
elif EXPORT_KIND == "olca":
    ndb.write_db_to_olca(filepath=str(EXPORT_ROOT / "olca"))
elif EXPORT_KIND == "datapackage":
    ndb.write_datapackage(name="premise-remind-tutorial")
else:
    raise ValueError(f"Unsupported EXPORT_KIND: {EXPORT_KIND}")


## Which format should you choose?

| Target | Use it when | Portability note |
|---|---|---|
| Brightway databases | Analysis stays in the active project | Database IDs and project state are local |
| Superstructure | Comparing scenarios in Activity Browser | Share the difference file with the matching union database |
| SimaPro / OpenLCA CSV | Importing into those desktop tools | Review category and flow mappings after import |
| Premise datapackage | Sharing scenario databases between licensed ecoinvent users | Recipients still need a compatible local ecoinvent copy |

The screenshot below shows a superstructure analysis setup in Activity Browser.

![Superstructure analysis in Activity Browser](assets/example_superstructure.png)


## Pitfalls and extension

- The number of Brightway output names must equal the number of scenarios.
- A superstructure needs at least two distinct scenarios.
- Inspect unmatched SimaPro/OpenLCA flows after export.
- Extension: use notebook 07 for a modern Brightway scenario-array ZIP instead of a scenario-difference file.

## Exercise

Choose the export that best fits a collaborator who has ecoinvent but does not share your Brightway project.


In [ ]:
exercise_choice = "datapackage"
exercise_reason = "Portable reconstruction with a compatible local ecoinvent copy."
